In [ ]:
# COMPREHENSIVE ANALYSIS NOTEBOOK
# Loads all 6 master_results.json files and produces
# every plot and table needed for the report



import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns
from google.colab import drive
import os

drive.mount('/content/drive')

# ── Style ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.grid':          True,
    'grid.color':         '#E5E7EB',
    'grid.linewidth':     0.6,
    'axes.labelsize':     12,
    'axes.titlesize':     13,
    'axes.titleweight':   'bold',
    'xtick.labelsize':    11,
    'ytick.labelsize':    11,
    'legend.fontsize':    10,
    'legend.framealpha':  0.9,
    'figure.dpi':         150,
})

# ── Color palette ─────────────────────────────────────────────────────────
COLORS = {
    'baseline':      '#6B7280',  # gray
    'linear':        '#2563EB',  # blue
    'multiplicative':'#F59E0B',  # amber
    'lowrank':       '#7C3AED',  # purple
    'crf':           '#DC2626',  # red
    'attention':     '#059669',  # emerald
    'residual_mlp':  '#EA580C',  # orange
}

MARKERS = {
    'baseline':       'o',
    'linear':         's',
    'multiplicative': '^',
    'crf':            '*',
    'residual_mlp':   'X',
}

LABELS = {
    'linear':         'Linear Additive',
    'multiplicative': 'Multiplicative',
    'crf':            'CRF (Symmetric)',
    'residual_mlp':   'Residual MLP',
}

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DRIVE = '../../results/t2d/'
EXPERIMENTS = {
    'linear':         'linear_additive_master_results.json',
    'multiplicative': 'multiplicative_master_results.json',
    'crf':            'crf_master_results.json',
    'residual_mlp':   'residual_mlp_master_results.json',
    'combined':       'combined_master_results.json',
}

SEEDS            = [42, 123, 456, 789, 1337, 2024, 9999]
FRACTIONS        = [1.0, 0.8, 0.6, 0.4, 0.2]
FRACTION_LABELS  = ['100%', '80%', '60%', '40%', '20%']

# ── Output directory ───────────────────────────────────────────────────────
OUT_DIR = '../../plots/'
os.makedirs(OUT_DIR, exist_ok=True)

print(" Setup complete.")
print(f"Output directory: {OUT_DIR}")




In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns
from google.colab import drive
import os

drive.mount('/content/drive')

# ── Style ─────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.grid':          True,
    'grid.color':         '#E5E7EB',
    'grid.linewidth':     0.6,
    'axes.labelsize':     12,
    'axes.titlesize':     13,
    'axes.titleweight':   'bold',
    'xtick.labelsize':    11,
    'ytick.labelsize':    11,
    'legend.fontsize':    10,
    'legend.framealpha':  0.9,
    'figure.dpi':         150,
})

# ── Color palette ─────────────────────────────────────────────
COLORS = {
    'baseline':      '#6B7280',  # gray
    'linear':        '#2563EB',  # blue
    'multiplicative':'#F59E0B',  # amber
    'lowrank':       '#7C3AED',  # purple
    'crf':           '#DC2626',  # red
    'attention':     '#059669',  # emerald
    'residual_mlp':  '#EA580C',  # orange
}

MARKERS = {
    'baseline':       'o',
    'linear':         's',
    'multiplicative': '^',
    'crf':            '*',
    'residual_mlp':   'X',
}

LABELS = {
    'linear':         'Linear Additive',
    'multiplicative': 'Multiplicative',
    'crf':            'CRF (Symmetric)',
    'residual_mlp':   'Residual MLP',
}

# ── Paths ──────────────────────────────────────────────
BASE_DRIVE = '../../results/t2d/'
EXPERIMENTS = {
    'linear':         'linear_additive_master_results.json',
    'multiplicative': 'multiplicative_master_results.json',
    'crf':            'crf_master_results.json',
    'residual_mlp':   'residual_mlp_master_results.json',
    'combined':       'combined_master_results.json',
}

SEEDS            = [42, 123, 456, 789, 1337, 2024, 9999]
FRACTIONS        = [1.0, 0.8, 0.6, 0.4, 0.2]
FRACTION_LABELS  = ['100%', '80%', '60%', '40%', '20%']

# ── Output directory ────────────────────────────────────────
OUT_DIR = '../../plots/'
os.makedirs(OUT_DIR, exist_ok=True)

print(" Setup complete.")
print(f"Output directory: {OUT_DIR}")





def load_results(exp_key, interaction_key):
    """Load master_results.json for a given experiment."""
    path = os.path.join(BASE_DRIVE, EXPERIMENTS[exp_key])
    with open(path, 'r') as f:
        master = json.load(f)
    return master['results']

# Store all data
all_data = {}
for key in INTERACTION_KEYS:
    all_data[key] = load_results(key, key)
    print(f" Loaded: {LABELS[key]}")

def get_auroc(data, interaction_key, seed, fraction):
    """Get interaction model AUROC for a given seed and fraction."""
    # Adjust the interaction_key if it's 'linear' to match the actual key in the JSON
    if interaction_key == 'linear':
        return data[str(seed)][str(fraction)]['linear_additive']['auroc_macro']
    else:
        return data[str(seed)][str(fraction)][interaction_key]['auroc_macro']

def get_base_auroc(data, seed, fraction):
    """Get baseline AUROC for a given seed and fraction."""
    return data[str(seed)][str(fraction)]['baseline']['auroc_macro']

def get_gap(data, seed, fraction):
    """Get gap for a given seed and fraction."""
    return data[str(seed)][str(fraction)]['gap']

def aggregate(data, interaction_key, fraction):
    """Compute mean and std across seeds for a given fraction."""
    aurocs = [get_auroc(data, interaction_key, s, fraction) for s in SEEDS]
    bases  = [get_base_auroc(data, s, fraction) for s in SEEDS]
    gaps   = [get_gap(data, s, fraction) for s in SEEDS]
    return {
        'auroc_mean': np.mean(aurocs), 'auroc_std': np.std(aurocs),
        'base_mean':  np.mean(bases),  'base_std':  np.std(bases),
        'gap_mean':   np.mean(gaps),   'gap_std':   np.std(gaps),
    }

print("\n All results loaded successfully.")


In [ ]:

print("=" * 90)
print("TABLE 1 — FULL SIX-WAY COMPARISON (mean ± std across 3 seeds)")
print("=" * 90)
print(f"\n{'Interaction':<22} {'100%':<20} {'80%':<20} {'60%':<20} {'40%':<20} {'20%':<20}")
print("-" * 110)

# Baseline first
base_row = []
for f in FRACTIONS:
    bases = [get_base_auroc(all_data['linear'], s, f) for s in SEEDS]
    base_row.append(f"{np.mean(bases):.4f}±{np.std(bases):.4f}")
print(f"{'Baseline':<22} " + "  ".join(f"{v:<18}" for v in base_row))
print()

for key in INTERACTION_KEYS:
    row = []
    for f in FRACTIONS:
        agg = aggregate(all_data[key], key, f)
        row.append(f"{agg['auroc_mean']:.4f}±{agg['auroc_std']:.4f}")
    print(f"{LABELS[key]:<22} " + "  ".join(f"{v:<18}" for v in row))

print("\n" + "=" * 90)
print("TABLE 2 — GAP TABLE (Δ AUROC = Interaction − Baseline, mean ± std)")
print("=" * 90)
print(f"\n{'Interaction':<22} {'100%':<20} {'80%':<20} {'60%':<20} {'40%':<20} {'20%':<20}")
print("-" * 110)

for key in INTERACTION_KEYS:
    row = []
    for f in FRACTIONS:
        agg = aggregate(all_data[key], key, f)
        g   = agg['gap_mean']
        gs  = agg['gap_std']
        tag = '' if g > 0 else ''
        row.append(f"{g:+.4f}±{gs:.4f}{tag}")
    print(f"{LABELS[key]:<22} " + "  ".join(f"{v:<20}" for v in row))

print("\n" + "=" * 90)
print("TABLE 3 — WINNER BY FRACTION")
print("=" * 90)
print(f"\n{'Fraction':<12} {'Winner':<25} {'Gap':<12} {'Runner-up':<25} {'Gap'}")
print("-" * 80)

for i, f in enumerate(FRACTIONS):
    gaps = {}
    for key in INTERACTION_KEYS:
        agg = aggregate(all_data[key], key, f)
        gaps[key] = agg['gap_mean']
    sorted_gaps = sorted(gaps.items(), key=lambda x: x[1], reverse=True)
    winner      = sorted_gaps[0]
    runner_up   = sorted_gaps[1]
    print(f"{FRACTION_LABELS[i]:<12} {LABELS[winner[0]]:<25} {winner[1]:+.4f}      "
          f"{LABELS[runner_up[0]]:<25} {runner_up[1]:+.4f}")



















In [ ]:



plt.close('all')
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('AUROC vs Training Data Fraction — All Interaction Types',
             fontsize=15, fontweight='bold', y=1.02)

x_pos = np.arange(len(FRACTIONS))

# ── Left: Absolute AUROC ──────────────────────────────────────────────────
ax = axes[0]

# Baseline
base_means = [np.mean([get_base_auroc(all_data['linear'], s, f) for s in SEEDS])
              for f in FRACTIONS]
base_stds  = [np.std([get_base_auroc(all_data['linear'], s, f) for s in SEEDS])
              for f in FRACTIONS]

# Per-seed faint lines for baseline
for seed in SEEDS:
    vals = [get_base_auroc(all_data['linear'], seed, f) for f in FRACTIONS]
    ax.plot(x_pos, vals, 'o--', color=COLORS['baseline'],
            linewidth=0.6, markersize=3, alpha=0.25)

ax.errorbar(x_pos, base_means, yerr=base_stds, fmt='o-',
            color=COLORS['baseline'], linewidth=2.5, markersize=8,
            capsize=4, label='Baseline', zorder=5)

# Each interaction
for key in INTERACTION_KEYS:
    means = [aggregate(all_data[key], key, f)['auroc_mean'] for f in FRACTIONS]
    stds  = [aggregate(all_data[key], key, f)['auroc_std']  for f in FRACTIONS]

    for seed in SEEDS:
        vals = [get_auroc(all_data[key], key, seed, f) for f in FRACTIONS]
        ax.plot(x_pos, vals, '--', color=COLORS[key],
                linewidth=0.5, alpha=0.2)

    ax.errorbar(x_pos, means, yerr=stds,
                fmt=f'{MARKERS[key]}-',
                color=COLORS[key], linewidth=2,
                markersize=8 if key == 'crf' else 7,
                capsize=4, label=LABELS[key])

ax.set_xticks(x_pos)
ax.set_xticklabels(FRACTION_LABELS)
ax.set_xlabel('Training Data Fraction')
ax.set_ylabel('Test AUROC (macro)')
ax.set_title('Absolute AUROC')
ax.set_ylim(0.52, 0.72)
ax.legend(loc='lower right', fontsize=9)

# ── Right: Gap (Interaction − Baseline) ──────────────────────────────────
ax = axes[1]
ax.axhline(y=0, color='black', linewidth=1.2, linestyle='--', alpha=0.5)

for key in INTERACTION_KEYS:
    means = [aggregate(all_data[key], key, f)['gap_mean'] for f in FRACTIONS]
    stds  = [aggregate(all_data[key], key, f)['gap_std']  for f in FRACTIONS]
    ax.errorbar(x_pos, means, yerr=stds,
                fmt=f'{MARKERS[key]}-',
                color=COLORS[key], linewidth=2,
                markersize=8 if key == 'crf' else 7,
                capsize=4, label=LABELS[key])

ax.set_xticks(x_pos)
ax.set_xticklabels(FRACTION_LABELS)
ax.set_xlabel('Training Data Fraction')
ax.set_ylabel('Δ AUROC (Interaction − Baseline)')
ax.set_title('Gap vs Baseline')
ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()
path = os.path.join(OUT_DIR, 'plot1_auroc_vs_fraction.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f" Plot 1 saved: {path}")

In [ ]:


plt.close('all')
fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle('Gap Heatmap — Δ AUROC (Interaction − Baseline)',
             fontsize=14, fontweight='bold')

gap_matrix = np.zeros((len(INTERACTION_KEYS), len(FRACTIONS)))
for i, key in enumerate(INTERACTION_KEYS):
    for j, f in enumerate(FRACTIONS):
        gap_matrix[i, j] = aggregate(all_data[key], key, f)['gap_mean']

vmax = max(abs(gap_matrix.min()), abs(gap_matrix.max()))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
im   = ax.imshow(gap_matrix, cmap='RdYlGn', norm=norm, aspect='auto')

ax.set_xticks(range(len(FRACTIONS)))
ax.set_xticklabels(FRACTION_LABELS, fontsize=12)
ax.set_yticks(range(len(INTERACTION_KEYS)))
ax.set_yticklabels([LABELS[k] for k in INTERACTION_KEYS], fontsize=11)
ax.set_xlabel('Training Data Fraction', fontsize=12)
ax.grid(False)

for i in range(len(INTERACTION_KEYS)):
    for j in range(len(FRACTIONS)):
        val   = gap_matrix[i, j]
        color = 'white' if abs(val) > vmax * 0.6 else 'black'
        std   = aggregate(all_data[INTERACTION_KEYS[i]], INTERACTION_KEYS[i],
                         FRACTIONS[j])['gap_std']
        ax.text(j, i, f'{val:+.4f}\n±{std:.4f}',
                ha='center', va='center',
                fontsize=8.5, fontweight='bold', color=color)

cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Δ AUROC (mean across seeds)', fontsize=11)

plt.tight_layout()
path = os.path.join(OUT_DIR, 'plot2_gap_heatmap.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f" Plot 2 saved: {path}")

In [ ]:

plt.close('all')
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('Winner by Data Fraction — Gap Magnitude',
             fontsize=14, fontweight='bold')

x_pos = np.arange(len(FRACTIONS))
width = 0.13
offsets = np.linspace(-0.35, 0.35, len(INTERACTION_KEYS))

for idx, key in enumerate(INTERACTION_KEYS):
    gaps = [aggregate(all_data[key], key, f)['gap_mean'] for f in FRACTIONS]
    stds = [aggregate(all_data[key], key, f)['gap_std']  for f in FRACTIONS]
    ax.bar(x_pos + offsets[idx], gaps, width,
           color=COLORS[key], alpha=0.85,
           label=LABELS[key], yerr=stds,
           capsize=3, error_kw={'linewidth': 1})

ax.axhline(y=0, color='black', linewidth=1, linestyle='--')
ax.set_xticks(x_pos)
ax.set_xticklabels(FRACTION_LABELS)
ax.set_xlabel('Training Data Fraction')
ax.set_ylabel('Δ AUROC (mean ± std)')
ax.set_title('All Interaction Types — Gap vs Baseline')
ax.legend(loc='upper right', fontsize=9, ncol=2)

plt.tight_layout()
path = os.path.join(OUT_DIR, 'plot3_winner_by_fraction.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f" Plot 3 saved: {path}")

In [ ]:

plt.close('all')
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Per-Seed Distribution — Gap vs Baseline (all interactions)',
             fontsize=14, fontweight='bold')

axes = axes.flatten()
x_pos = np.arange(len(FRACTIONS))

for idx, key in enumerate(INTERACTION_KEYS):
    ax = axes[idx]
    ax.axhline(y=0, color='black', linewidth=1, linestyle='--', alpha=0.5)

    seed_colors = ['#1D4ED8', '#7C3AED', '#B45309']
    for si, seed in enumerate(SEEDS):
        gaps = [get_gap(all_data[key], seed, f) for f in FRACTIONS]
        ax.plot(x_pos, gaps, 'o-', color=seed_colors[si],
                linewidth=1.5, markersize=6,
                label=f'Seed {seed}', alpha=0.8)

    means = [aggregate(all_data[key], key, f)['gap_mean'] for f in FRACTIONS]
    stds  = [aggregate(all_data[key], key, f)['gap_std']  for f in FRACTIONS]
    ax.errorbar(x_pos, means, yerr=stds, fmt='s-',
                color=COLORS[key], linewidth=2.5,
                markersize=9, capsize=4,
                label='Mean ± std', zorder=5)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(FRACTION_LABELS)
    ax.set_title(LABELS[key], color=COLORS[key])
    ax.set_ylabel('Δ AUROC')
    ax.set_xlabel('Data Fraction')
    ax.legend(fontsize=8)

plt.tight_layout()
path = os.path.join(OUT_DIR, 'plot4_per_seed_distribution.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f" Plot 4 saved: {path}")

In [ ]:

plt.close('all')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CRF vs Residual MLP — The Theoretical Result\n'
             '3 parameters vs 123 parameters',
             fontsize=14, fontweight='bold')

x_pos = np.arange(len(FRACTIONS))

# ── Left: Direct gap comparison ───────────────────────────────────────────
ax = axes[0]
ax.axhline(y=0, color='black', linewidth=1, linestyle='--', alpha=0.5)

for key, label in [('crf', 'CRF (3 params)'), ('residual_mlp', 'Residual MLP (123 params)')]:
    means = [aggregate(all_data[key], key, f)['gap_mean'] for f in FRACTIONS]
    stds  = [aggregate(all_data[key], key, f)['gap_std']  for f in FRACTIONS]
    ax.errorbar(x_pos, means, yerr=stds,
                fmt=f'{MARKERS[key]}-',
                color=COLORS[key], linewidth=2.5,
                markersize=9, capsize=4, label=label)

    for seed in SEEDS:
        gaps = [get_gap(all_data[key], seed, f) for f in FRACTIONS]
        ax.plot(x_pos, gaps, '--', color=COLORS[key],
                linewidth=0.6, alpha=0.25)

ax.set_xticks(x_pos)
ax.set_xticklabels(FRACTION_LABELS)
ax.set_xlabel('Training Data Fraction')
ax.set_ylabel('Δ AUROC (mean ± std)')
ax.set_title('Gap vs Baseline')
ax.legend()

# ── Right: CRF advantage over ResMLP ─────────────────────────────────────
ax = axes[1]
ax.axhline(y=0, color='black', linewidth=1, linestyle='--', alpha=0.5)

crf_advantages = []
for f in FRACTIONS:
    crf_gap  = aggregate(all_data['crf'],         'crf',         f)['gap_mean']
    rmlp_gap = aggregate(all_data['residual_mlp'], 'residual_mlp', f)['gap_mean']
    crf_advantages.append(crf_gap - rmlp_gap)

colors_adv = [COLORS['crf'] if v >= 0 else COLORS['residual_mlp']
              for v in crf_advantages]
bars = ax.bar(x_pos, crf_advantages, color=colors_adv, alpha=0.85,
              edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, crf_advantages):
    ypos = bar.get_height() + 0.001 if val >= 0 else bar.get_height() - 0.003
    ax.text(bar.get_x() + bar.get_width()/2, ypos,
            f'{val:+.4f}', ha='center', va='bottom',
            fontsize=9, fontweight='bold')

ax.set_xticks(x_pos)
ax.set_xticklabels(FRACTION_LABELS)
ax.set_xlabel('Training Data Fraction')
ax.set_ylabel('CRF Gap − ResMLP Gap')
ax.set_title('CRF Advantage over Residual MLP\n(positive = CRF wins)')

crf_patch  = mpatches.Patch(color=COLORS['crf'],         label='CRF wins')
rmlp_patch = mpatches.Patch(color=COLORS['residual_mlp'], label='ResMLP wins')
ax.legend(handles=[crf_patch, rmlp_patch])

plt.tight_layout()
path = os.path.join(OUT_DIR, 'plot5_crf_vs_residual_mlp.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f" Plot 5 saved: {path}")

In [ ]:

plt.close('all')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CRF — Learned Beta (Co-occurrence) Weights',
             fontsize=14, fontweight='bold')

PAIRS       = ['NEP-NEU', 'NEP-RET', 'NEU-RET']
EMPIRICAL   = [0.218, 0.247, 0.245]  # empirical co-occurrence rates

# Load beta weights from saved files
beta_by_seed_fraction = {}
for seed in SEEDS:
    for f in FRACTIONS:
        pct = f'{int(f*100)}pct'
        path_b = (f'/content/drive/MyDrive/comorbidity_experiments/'
                  f'subset_crf_v3/seed_{seed}/{pct}/beta_matrix.npy')
        try:
            beta_mat = np.load(path_b)
            # Extract upper triangle: (0,1)=NEP-NEU, (0,2)=NEP-RET, (1,2)=NEU-RET
            beta_by_seed_fraction[(seed, f)] = [
                beta_mat[0, 1], beta_mat[0, 2], beta_mat[1, 2]
            ]
        except FileNotFoundError:
            beta_by_seed_fraction[(seed, f)] = [np.nan, np.nan, np.nan]

# ── Left: Beta weights at 100% data across seeds ──────────────────────────
ax      = axes[0]
x_pairs = np.arange(len(PAIRS))
width   = 0.25
seed_colors = ['#1D4ED8', '#7C3AED', '#B45309']

for si, seed in enumerate(SEEDS):
    betas = beta_by_seed_fraction.get((seed, 1.0), [np.nan]*3)
    ax.bar(x_pairs + (si - 1) * width, betas, width,
           color=seed_colors[si], alpha=0.85,
           label=f'Seed {seed}')

ax.set_xticks(x_pairs)
ax.set_xticklabels(PAIRS)
ax.set_ylabel('Learned β (co-occurrence weight)')
ax.set_xlabel('Complication Pair')
ax.set_title('Beta Weights at 100% Data\n(consistency across seeds)')
ax.axhline(y=0, color='black', linewidth=0.8)
ax.legend()

# ── Right: Beta weights vs empirical co-occurrence ────────────────────────
ax = axes[1]

# Mean beta across seeds at 100% data
mean_betas = []
for pi in range(3):
    vals = [beta_by_seed_fraction.get((s, 1.0), [np.nan]*3)[pi]
            for s in SEEDS]
    mean_betas.append(np.nanmean(vals))

x = np.arange(len(PAIRS))
ax2 = ax.twinx()

bars = ax.bar(x - 0.2, mean_betas, 0.35,
              color=COLORS['crf'], alpha=0.8,
              label='Mean β (learned)')
ax2.bar(x + 0.2, EMPIRICAL, 0.35,
        color='#6B7280', alpha=0.6,
        label='Empirical co-occurrence rate')

ax.set_xticks(x)
ax.set_xticklabels(PAIRS)
ax.set_ylabel('Learned β weight', color=COLORS['crf'])
ax2.set_ylabel('Empirical co-occurrence rate', color='#6B7280')
ax.set_title('Learned Beta vs Empirical Co-occurrence\n(do they agree?)')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

plt.tight_layout()
path = os.path.join(OUT_DIR, 'plot6_crf_beta_analysis.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f" Plot 6 saved: {path}")

In [ ]:

print("\n" + "=" * 70)
print("COMPLETE NUMERICAL SUMMARY FOR REPORT")
print("=" * 70)

for key in INTERACTION_KEYS:
    print(f"\n── {LABELS[key]} ──")
    for f, fl in zip(FRACTIONS, FRACTION_LABELS):
        agg    = aggregate(all_data[key], key, f)
        winner = 'Interaction ' if agg['gap_mean'] > 0 else 'Baseline '
        sig    = '(std < mean)' if agg['gap_std'] < abs(agg['gap_mean']) else '(std ≥ mean)'
        print(f"  {fl}: base={agg['base_mean']:.4f}±{agg['base_std']:.4f}  "
              f"int={agg['auroc_mean']:.4f}±{agg['auroc_std']:.4f}  "
              f"gap={agg['gap_mean']:+.4f}±{agg['gap_std']:.4f}  "
              f"{winner} {sig}")

print("\n" + "=" * 70)
print("CRF BETA WEIGHTS SUMMARY (100% data)")
print("=" * 70)
for pi, pair in enumerate(PAIRS):
    vals = [beta_by_seed_fraction.get((s, 1.0), [np.nan]*3)[pi] for s in SEEDS]
    print(f"  {pair}: seed42={vals[0]:.4f}  seed123={vals[1]:.4f}  "
          f"seed456={vals[2]:.4f}  mean={np.nanmean(vals):.4f}")
print(f"\n  Empirical co-occurrence: NEP-NEU={EMPIRICAL[0]}  "
      f"NEP-RET={EMPIRICAL[1]}  NEU-RET={EMPIRICAL[2]}")

print("\n Analysis complete. All plots saved to Drive.")
print(f"   {OUT_DIR}")